In [ ]:
import torch
import pandas as pd
import re
import os
import unicodedata
from unidecode import unidecode
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path

# Como o notebook está em 'corretor/src/3_avaliacao' e o .env em 'corretor',
# precisamos voltar duas pastas (../..) para achar o arquivo.
env_path = Path('../../.env')

# Carrega o arquivo explicitamente
load_dotenv(dotenv_path=env_path)

In [ ]:
PATH_BASE_TESTES_FINAIS = os.getenv('PATH_BASE_TESTES_FINAIS')
PATH_GABARITO_CORRETOR = os.getenv('PATH_GABARITO_CORRETOR')
PATH_BASE_ENTROPIA_BERT = os.getenv('PATH_BASE_ENTROPIA_BERT')

In [ ]:
df_texts = pd.read_csv(PATH_BASE_TESTES_FINAIS)

In [ ]:
def preprocess(relato):
    def remove_stop_words(relato):
        sw = stopwords.words('portuguese')
        sw = [unidecode(word).upper() for word in sw]
        return ' '.join([token for token in re.split("[, ]", relato) if len(token) > 2 and token not in sw])
    
    def clean_relato(relato):
        relato = re.sub(r'\@(\w*\d?)', " ", relato)
        relato = re.sub(r'([A-z1-9]*_+[A-z1-9]*)|(_+[A-z1-9]*)|([A-z1-9]*_+)', " ", relato)
        relato = re.sub(r"((?:HTTPS?|WWW)(?:\:\/\/?)?[*a-zA-Z0-9\.\/\?\:@\-_=#]+[\.][a-zA-Z]{2,6}[a-zA-Z0-9\.\&\/\?\:@\-_=#]*)", " ", relato)
        relato = re.sub(r'(<[^<>]*>)|(&NBSP;)|(&nbsp;)', " ", relato)
        relato = re.sub(r"[^\w\s]", " ", relato)
        relato = re.sub(r"[\d]", " ", relato)
        relato = remove_stop_words(relato)
        return relato
    
    relato = clean_relato(relato)
    return relato

def normalize_text(text):
    if pd.isna(text):
        return text
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    return text.upper()
df_texts["relato_tratado"] = df_texts["relato"].apply(normalize_text).apply(preprocess)
df_texts

In [ ]:
df_texts["relato_tratado"] = df_texts["relato"].apply(normalize_text).apply(preprocess)
df_texts

In [ ]:
df_texts['relato_tratado'] = df_texts['relato_tratado'].str.lower()

In [ ]:
df_texts

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch
import pandas as pd

In [ ]:
model_name = "lfcc/bert-portuguese-ner"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

In [ ]:
def classify_tokens(text, max_words = 200):
    def split_text(text):
        words = text.split()
        chunks = [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]
        return chunks
        
    classified_tokens = []
    text_chunks = split_text(text)

    for chunk in text_chunks:
        inputs = tokenizer(chunk, return_tensors="pt", add_special_tokens=False)

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=2)

        label_list = model.config.id2label
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
        predictions = predictions[0].tolist()

        current_entity = []
        current_entity_type = None

        for token, prediction in zip(tokens, predictions):
            if token.startswith("##"):
                if current_entity:
                    current_entity.append(token[2:])
            else:
                if current_entity:
                    word = ''.join(current_entity)
                    label = 1 if "Pes" in current_entity_type else 0
                    classified_tokens.append({"palavras": word, "label": label})
                    current_entity = []

                current_entity = [token]
                current_entity_type = label_list[prediction]
                    
        if current_entity:
            token = ''.join(current_entity)
            label = 1 if "Pes" in current_entity_type else 0
            classified_tokens.append({"palavras": token, "label": label})
    return classified_tokens

In [ ]:
classified_tokens = []
for idx, text in df_texts["relato_tratado"].items():
    classified_tokens.extend(classify_tokens(text))
classified_tokens_df = pd.DataFrame(classified_tokens)

In [ ]:
classified_tokens_df_upper = classified_tokens_df.copy()

In [ ]:
classified_tokens_df_upper['palavras'] = classified_tokens_df_upper['palavras'].str.upper()

In [ ]:
gab = pd.read_csv(PATH_GABARITO_CORRETOR)

In [ ]:
teste2 = classified_tokens_df_upper.merge(gab[['palavra', 'y']], left_on='palavras', right_on='palavra', how='left')

teste2['y'] = teste2['y'].fillna(0).astype(int)

teste2.value_counts(subset=['y'])

# Metricas


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
cm = confusion_matrix(teste2['y'], teste2['label'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap='binary')
plt.title("Bertimbau Fine-Tuning NER pt-br")
plt.show()

In [ ]:
accuracy = accuracy_score(teste2['y'], teste2['label'])
accuracy

In [ ]:
precision = precision_score(teste2['y'], teste2['label'])
precision

In [ ]:
recall = recall_score(teste2['y'], teste2['label'])
recall

In [ ]:
f1 = f1_score(teste2['y'], teste2['label'])
f1

# Anonimização

In [ ]:
def anon_tokens(text, text_orig):
    classified_tokens = classify_tokens(text)
    for token in classified_tokens:
        if token["label"] == 1:
            text_orig = text_orig.replace(token["palavras"].upper(), "***")
    return text_orig

In [ ]:
anon_tokens(df_texts["relato_tratado"][0], df_texts["relato"][0])

In [ ]:
df_texts_anon = df_texts.copy()

In [ ]:
for index, row in df_texts_anon.iterrows():
    df_texts_anon.at[index, "relato_anon"] = anon_tokens(row["relato_tratado"], row["relato"])

In [ ]:
df_texts_anon

# entropy

In [ ]:
import math
import numpy as np
import pandas as pd
from scipy.stats import entropy
from collections import Counter
import matplotlib.pyplot as plt
import scipy.stats as stats

É realizado o cálculo de entropia dos textos em claro e anonimizados, após isso realizada a comparação dos resultados obtidos.

- Foi criada uma tabela com o resultado da entropia de cada texto e realizada
- Realizado o cálculo da média de cada conjunto: Média dos resultados dos textos em claro e média dos resultados dos textos anonimizados. Assim podendo quantificar o impacto.
- Também foi realizado o cálculo do desvio padrão para mensurar diferença das entropias.

A entropia é calulada com base na probabilidade da ocorrência de cada caracter no texto. O resultado obtido da probabilidade de cada caracter é inserido para calculo da entropia utilizando a biblioteca entropy do scipy.stats que calcula a entropia com base na formula da entropia de shannon ( entropia da informação ).


In [ ]:
def calculate_text_entropy(text, base=None):
    char_counts = Counter(text)
    total_chars = sum(char_counts.values())
    probabilities = np.array([count / total_chars for count in char_counts.values()])

    return entropy(probabilities, base=base)

resultado texto unicos

In [ ]:
text = df_texts_anon['relato'].iloc[0]
text_anon = df_texts_anon['relato_anon'].iloc[0]

entropia do texto original

In [ ]:
entropia = calculate_text_entropy(text, base=2)
entropia

entropia do texto anonimizado

In [ ]:
entropia = calculate_text_entropy(text_anon, base=2)
entropia

resultado sobre toda a base de BOs

In [ ]:
df_texts_anon['entrpy_result_orig'] = df_texts_anon['relato'].apply(lambda x: calculate_text_entropy(x, base=2))

In [ ]:
df_texts_anon['entrpy_result_anon'] = df_texts_anon['relato_anon'].apply(lambda x: calculate_text_entropy(x, base=2))

In [ ]:
save_df = df_texts_anon[["n_bop", "relato", "relato_anon", "entrpy_result_orig", "entrpy_result_anon"]]

In [ ]:
save_df = save_df.rename(columns={"entrpy_result_orig":"bertimbau_text_orig", "entrpy_result_anon":"Entropy Bertimbau"})

In [ ]:
save_df.to_csv(PATH_BASE_ENTROPIA_BERT, index=False)

### Verificando se há outliers

In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(df_texts_anon['entrpy_result_orig'], patch_artist=True)
plt.grid(True)
plt.title('Validação Cruzada')
plt.ylabel('Acurácia')
plt.show()


In [ ]:
plt.figure(figsize=(10, 4))
plt.boxplot(df_texts_anon['entrpy_result_anon'], patch_artist=True)
plt.grid(True)
plt.title('Validação Cruzada')
plt.ylabel('Acurácia')
plt.show()


#### Teste Shapiro para verificação da distribuição

In [ ]:
shapiro_orig = stats.shapiro(df_texts_anon["entrpy_result_orig"])
shapiro_anon = stats.shapiro(df_texts_anon["entrpy_result_anon"])
print(f"Teste de Shapiro-Wilk para entrpy_result_orig: p-valor={shapiro_orig.pvalue}")
print(f"Teste de Shapiro-Wilk para entrpy_result_anon: p-valor={shapiro_anon.pvalue}")

#### Teste de Wilcoxon - verificação da significancia

recomendação para utilização desse teste em casos que houvesse outliers - o que foi visto nas células acima

In [ ]:
w_stat, p_value_w = stats.wilcoxon(df_texts_anon["entrpy_result_orig"], df_texts_anon["entrpy_result_anon"])

In [ ]:
print(f"{w_stat} \n{p_value_w}")

#### Teste T - para calculo de significancia também

In [ ]:
t_stat, p_value_t = stats.ttest_rel(df_texts_anon["entrpy_result_orig"], df_texts_anon["entrpy_result_anon"])

In [ ]:
print(f"{w_stat} \n{p_value_w}")

In [ ]:
alpha = 0.05
if p_value_t < alpha:
    print("A diferença entre as entropias é estatisticamente significativa (Teste t).")
else:
    print("A diferença entre as entropias NÃO é estatisticamente significativa (Teste t).")

if p_value_w < alpha:
    print("A diferença entre as entropias é estatisticamente significativa (Teste de Wilcoxon).")
else:
    print("A diferença entre as entropias NÃO é estatisticamente significativa (Teste de Wilcoxon).")

Media dos valores

In [ ]:
df_texts_anon['entrpy_result_orig'].values.mean()

In [ ]:
df_texts_anon['entrpy_result_anon'].values.mean()

In [ ]:
np.std(df_texts_anon['entrpy_result_orig'].values)

In [ ]:
np.std(df_texts_anon['entrpy_result_anon'].values)